In [ ]:
!pip install -U openai

In [ ]:
import os
os.environ["HF_TOKEN"] = "" #Collect access token from Hugging Face
print(os.environ.get('HF_TOKEN'))

In [ ]:
import os
from openai import OpenAI
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

MODEL = "Qwen/Qwen3-VL-235B-A22B-Thinking:novita"

In [ ]:
from PIL import Image
import io, base64
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import os, csv, json, yaml, time
from pathlib import Path


OUTPUT_DIR = "generated_files"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SINGLE_CSV = os.path.join(OUTPUT_DIR, "mammogram_features.csv")


def encode_image(image_path):
    img = Image.open(image_path)
    img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)

    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def analyze_mammogram_and_save_single_csv(image_path: str):
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    img = encode_image(image_path)


    
    response = client.chat.completions.create(
        model=MODEL,
        messages = [
            {
                "role": "system",
                "content": SYSTEM_FEATURE_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": FEATURE_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{img}"
                        }
                    }
                ]
            }
        ],
        tools=tools,
        tool_choice={
            "type": "function", 
            "function": {"name": "mammogram_feature_extraction"}
        },
        temperature=0.0,
        top_p=0.1,
        extra_body={"top_k": 0},
    )

    
    tool_call = response.choices[0].message.tool_calls[0]
    print(tool_call)
    data = json.loads(tool_call.function.arguments)  

    
    data.setdefault("image_id", image_id)

    
    write_header = not os.path.exists(SINGLE_CSV)
    with open(SINGLE_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FEATURE_FIELDS)
        if write_header:
            writer.writeheader()
        row = {k: data.get(k, None) for k in FEATURE_FIELDS}
        writer.writerow(row)

    print(f"Processed {image_id} → saved to {SINGLE_CSV}")
    return row


def process_images_sequentially(input_dir, delay_seconds=3,
                                resume_from=1, log_file="processed_images.log"):

    input_path = Path(input_dir)
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    # ✅ Load log file
    log_path = output_path / log_file
    processed_ids = set()

    if log_path.exists():
        with open(log_path, "r", encoding="utf-8") as lf:
            processed_ids = {line.strip() for line in lf if line.strip()}
        print(f"Resuming… {len(processed_ids)} images already processed.")

    # ✅ Collect all PNG files in directory (recursively)
    png_files = sorted(list(input_path.rglob("*.png")))
    print(f"Total PNG images found: {len(png_files)}")

    for idx, img_path in enumerate(png_files, start=1):
        if idx < resume_from:
            continue

        image_id = img_path.stem  

        if image_id in processed_ids:
            print(f"[{idx}] Already processed: {image_id}")
            continue

        print(f"[{idx}] Processing: {img_path}")

        try:
            analyze_mammogram_and_save_single_csv(str(img_path))
            print(f"   Waiting {delay_seconds} seconds...")
            time.sleep(delay_seconds)

            # ✅ Write to log
            with open(log_path, "a", encoding="utf-8") as lf:
                lf.write(f"{image_id}\n")

        except Exception as e:
            print(f"[{idx}] Error processing {img_path}: {e}")


if __name__ == "__main__":
    INPUT_DIR = "" #Set location to dataset folder containing png files 
    LOG_FILE_NAME = "processed_images.log"

    process_images_sequentially(
        INPUT_DIR,
        delay_seconds=2,
        resume_from=1,
        log_file=LOG_FILE_NAME
    )

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    balanced_accuracy_score, 
    classification_report
)


train_csv = ""
generated_csv = "/kaggle/working/generated_files/mammogram_features.csv"


train_df = pd.read_csv(train_csv, usecols=["image_id", "cancer"])
gen_df = pd.read_csv(generated_csv, usecols=["image_id", "malignant"])

merged = pd.merge(train_df, gen_df, on="image_id", how="inner")

y_true = merged["cancer"].astype(int)
y_pred = merged["malignant"].map({"no": 0, "yes": 1}).astype(int)

# --- Confusion matrix ---
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

# --- Plot heatmap ---
plt.figure(figsize=(6,5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted Negative", "Predicted Positive"],
    yticklabels=["Actual Negative", "Actual Positive"]
)
plt.title("Confusion Matrix (Cancer vs Malignant) – Gemma")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix_gemini.png", dpi=300)
plt.show()

print("Confusion Matrix:")
print(cm)

# ============================================================
# ADDITIONAL METRICS
# ============================================================

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)         # Sensitivity
f1 = f1_score(y_true, y_pred, zero_division=0)
bal_acc = balanced_accuracy_score(y_true, y_pred)

# Specificity = TN / (TN + FP)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print("\n===== PERFORMANCE METRICS =====")
print(f"Accuracy:              {acc:.4f}")
print(f"Precision:             {prec:.4f}")
print(f"Recall (Sensitivity):  {rec:.4f}")
print(f"Specificity:           {specificity:.4f}")
print(f"F1 Score:              {f1:.4f}")
print(f"Balanced Accuracy:     {bal_acc:.4f}")

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"]))
